# IFRS S1/S2 Writer Stage

Consumes `generation_blocks_<bank>.json` + `evidence_store_<bank>.json` + `section_plan_<bank>.json`
and produces the aligned report. Per block: build prompt → call Azure → parse the JSON contract →
**deterministic gate** (re-resolve every cited `citation_id` to the store and check the number matches)
→ bounded revision on failure → assemble into section > subsection > block.

Traceability is enforced by code, not trusted: a hallucinated or wrong number cannot pass the gate.

In [ ]:
import os, re, json, time
from pathlib import Path
from collections import defaultdict

try:
    import requests
except ImportError:
    requests = None

OUT = Path("mapping_outputs")          # where the mapper wrote its artifacts
BANK = "BANK01"
MOCK_MODE = False                     # True = offline dry-run; False = real Azure calls
JSON_MODE = True                      # requests JSON output when supported
TEMPERATURE = 0.2
MAX_TOKENS = 2000
MAX_REVISIONS = 2


# ---------------------------------------------------------------------------
# Azure configuration
# ---------------------------------------------------------------------------
# The notebook uses the same environment variables as the platform:
#
# AZURE_OPENAI_FAST_DEPLOYMENT_URL=
#   https://<resource>.openai.azure.com/openai/deployments/<deployment>/
#   chat/completions?api-version=<api-version>
#
# AZURE_OPENAI_API_KEY=<key>
#
# The URL must be the COMPLETE chat-completions deployment URL.

def _find_env_file(filename=".env"):
    """Find .env in the current directory or one of its parent directories."""
    current = Path.cwd().resolve()

    for folder in (current, *current.parents):
        candidate = folder / filename
        if candidate.exists():
            return candidate

    return None


def _load_env_file(path):
    """
    Load a simple .env file and overwrite stale values already held by the
    notebook kernel. This matters when .env is edited without restarting the
    kernel.
    """
    if path is None:
        return

    with path.open("r", encoding="utf-8") as env_file:
        for raw_line in env_file:
            line = raw_line.strip()

            if (
                not line
                or line.startswith("#")
                or "=" not in line
            ):
                continue

            key, value = line.split("=", 1)
            key = key.strip()
            value = value.strip().strip('"').strip("'")

            if key:
                os.environ[key] = value


ENV_PATH = _find_env_file()
_load_env_file(ENV_PATH)

# Prefer the fast deployment variable used by the ESG platform. Older variable
# names remain as fallbacks so the notebook is backward compatible.
AZURE_URL = (
    os.getenv("AZURE_OPENAI_FAST_DEPLOYMENT_URL")
    or os.getenv("AZURE_OPENAI_URL")
    or os.getenv("AZURE_OPENAI_CHAT_URL")
    or os.getenv("OPENAI_URL")
)

AZURE_KEY = (
    os.getenv("AZURE_OPENAI_API_KEY")
    or os.getenv("AZURE_OPENAI_KEY")
    or os.getenv("OPENAI_API_KEY")
    or os.getenv("API_KEY")
)


def _validate_azure_configuration():
    if MOCK_MODE:
        return

    if not AZURE_URL:
        raise RuntimeError(
            "AZURE_OPENAI_FAST_DEPLOYMENT_URL is missing. "
            "Add the complete Azure chat-completions deployment URL to .env."
        )

    if not AZURE_KEY:
        raise RuntimeError(
            "AZURE_OPENAI_API_KEY is missing from .env."
        )

    if not AZURE_URL.startswith("https://"):
        raise RuntimeError(
            "AZURE_OPENAI_FAST_DEPLOYMENT_URL must begin with https://"
        )

    if "/chat/completions" not in AZURE_URL:
        raise RuntimeError(
            "AZURE_OPENAI_FAST_DEPLOYMENT_URL must be the complete deployment "
            "URL and include /chat/completions?api-version=..."
        )

    if "api-version=" not in AZURE_URL:
        raise RuntimeError(
            "AZURE_OPENAI_FAST_DEPLOYMENT_URL must include ?api-version=..."
        )

    if requests is None:
        raise ImportError(
            "The requests package is required. Install it with: pip install requests"
        )


_validate_azure_configuration()

print(
    "env file:",
    str(ENV_PATH) if ENV_PATH else "(not found)",
)
print(
    "endpoint:",
    (AZURE_URL[:80] + "...") if AZURE_URL else "(mock)",
)
print(
    "key:",
    "set" if AZURE_KEY else "none",
    "| MOCK_MODE:",
    MOCK_MODE,
)


In [ ]:
def azure_chat(
    messages,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
    json_mode=JSON_MODE,
    retries=4,
    timeout=120,
):
    """
    POST to AZURE_OPENAI_FAST_DEPLOYMENT_URL using the Azure ``api-key`` header.

    Compatibility behavior:
    - tries ``max_completion_tokens`` first;
    - falls back to ``max_tokens`` if the deployment rejects it;
    - retries transient Azure/network failures;
    - retries without ``temperature`` if the model does not support it;
    - retries without ``response_format`` if JSON mode is unsupported.

    Returns the assistant message content as a string.
    """
    if MOCK_MODE:
        return _mock_chat(messages)

    if requests is None:
        raise ImportError(
            "The requests package is required. Install it with: pip install requests"
        )

    headers = {
        "api-key": AZURE_KEY,
        "Content-Type": "application/json",
        "Accept": "application/json",
    }

    last_error = None

    for token_field in (
        "max_completion_tokens",
        "max_tokens",
    ):
        include_temperature = temperature is not None
        include_json_mode = bool(json_mode)

        # Compatibility changes restart the request using the same token field.
        while True:
            compatibility_retry = False

            for attempt in range(retries):
                body = {
                    "messages": messages,
                    token_field: max_tokens,
                }

                if include_temperature:
                    body["temperature"] = temperature

                if include_json_mode:
                    body["response_format"] = {
                        "type": "json_object"
                    }

                try:
                    response = requests.post(
                        AZURE_URL,
                        headers=headers,
                        json=body,
                        timeout=timeout,
                    )

                except requests.RequestException as exc:
                    last_error = exc

                    if attempt < retries - 1:
                        wait = min(2 ** attempt, 10)
                        print(
                            f"Azure connection error; retrying in {wait}s "
                            f"({attempt + 1}/{retries})"
                        )
                        time.sleep(wait)
                        continue

                    break

                if response.status_code == 200:
                    try:
                        payload = response.json()
                        return payload["choices"][0]["message"]["content"]
                    except (
                        ValueError,
                        KeyError,
                        IndexError,
                        TypeError,
                    ) as exc:
                        raise RuntimeError(
                            "Azure returned an unexpected successful response: "
                            f"{response.text[:1000]}"
                        ) from exc

                error_text = response.text
                error_lower = error_text.lower()

                # Some deployments only accept the default temperature.
                if (
                    include_temperature
                    and response.status_code in (400, 422)
                    and "temperature" in error_lower
                ):
                    include_temperature = False
                    compatibility_retry = True
                    print(
                        "Azure rejected temperature; retrying without it."
                    )
                    break

                # Some API versions/models do not support JSON response mode.
                if (
                    include_json_mode
                    and response.status_code in (400, 422)
                    and (
                        "response_format" in error_lower
                        or "json_object" in error_lower
                    )
                ):
                    include_json_mode = False
                    compatibility_retry = True
                    print(
                        "Azure rejected response_format; retrying without JSON mode."
                    )
                    break

                # Try max_tokens when max_completion_tokens is unsupported.
                if (
                    token_field == "max_completion_tokens"
                    and response.status_code in (400, 422, 500)
                    and (
                        "max_completion_tokens" in error_lower
                        or "unsupported" in error_lower
                        or "invalid parameter" in error_lower
                    )
                ):
                    last_error = RuntimeError(
                        "Azure rejected max_completion_tokens: "
                        f"{error_text[:600]}"
                    )
                    print(
                        "Azure rejected max_completion_tokens; trying max_tokens."
                    )
                    break

                if response.status_code in (
                    429,
                    500,
                    502,
                    503,
                    504,
                ):
                    last_error = RuntimeError(
                        f"Azure HTTP {response.status_code}: "
                        f"{error_text[:600]}"
                    )

                    if attempt < retries - 1:
                        retry_after = response.headers.get(
                            "Retry-After"
                        )

                        try:
                            wait = (
                                float(retry_after)
                                if retry_after is not None
                                else min(2 ** attempt, 10)
                            )
                        except ValueError:
                            wait = min(2 ** attempt, 10)

                        print(
                            f"Azure HTTP {response.status_code}; "
                            f"retrying in {wait}s "
                            f"({attempt + 1}/{retries})"
                        )
                        time.sleep(wait)
                        continue

                    break

                raise RuntimeError(
                    f"Azure request failed with HTTP {response.status_code}: "
                    f"{error_text[:1200]}"
                )

            if compatibility_retry:
                continue

            # Move from max_completion_tokens to max_tokens.
            break

    raise RuntimeError(
        "Azure request failed after all retries and compatibility fallbacks. "
        f"Last error: {last_error}"
    )


def parse_contract(raw):
    s = raw.strip()

    if s.startswith("```"):
        s = re.sub(
            r"^```[a-zA-Z]*\n?",
            "",
            s,
        )
        s = re.sub(
            r"\n?```$",
            "",
            s,
        )

    return json.loads(s)


In [ ]:
# Mock Azure client: cites REAL ids/values from the prompt so the gate exercises the true path.
def _mock_chat(messages):
    user = messages[1]["content"] if len(messages) > 1 else messages[0]["content"]
    is_revision = messages[-1]["content"].startswith(("Fix ALL", "That was not"))
    mode = ("narrative" if "MODE: narrative" in user else "absence" if "MODE: absence" in user else "data_backed")
    reqids = re.findall(r"\[([A-Z0-9_]+)\]", user.split("EVIDENCE")[0])
    ev = re.findall(r"- (E-[A-Z0-9-]+): ([^\n(]+)", user)
    if mode == "narrative" or not ev:
        return json.dumps({"prose": "The entity confirms compliance with the applicable disclosure requirements.",
            "citations_used": [], "numeric_claims": [], "requirements_addressed": reqids[:6],
            "standards_covered": ["IFRS S1"]})
    picks = ev[:2]
    claims, cids, frags = [], [], []
    for cid, val in picks:
        n = re.search(r"-?\d[\d,]*\.?\d*", val)
        cids.append(cid)
        if n:
            claims.append({"text": n.group(0), "citation_id": cid})
            frags.append(f"{n.group(0)} [{cid}]")
        else:
            frags.append(f"the disclosed value [{cid}]")
    return json.dumps({"prose": "For the reporting period, " + "; ".join(frags) + ".",
        "citations_used": cids, "numeric_claims": claims,
        "requirements_addressed": reqids[:8], "standards_covered": ["IFRS S1", "IFRS S2"]})
print("mock client defined (used when MOCK_MODE=True)")

In [ ]:
blocks = json.loads((OUT / f"generation_blocks_{BANK}.json").read_text(encoding="utf-8"))
store  = json.loads((OUT / f"evidence_store_{BANK}.json").read_text(encoding="utf-8"))
plan   = json.loads((OUT / f"section_plan_{BANK}.json").read_text(encoding="utf-8"))
blocks_by_id = {b["block_id"]: b for b in blocks}
SECTION_TITLE = {s["section_key"]: s["section_title"] for s in plan["sections"]}

def deref(path):
    cur = store
    for p in path.strip("/").split("/"):
        cur = cur[p.replace("~1", "/").replace("~0", "~")]
    return cur
print(f"{len(blocks)} blocks | {sum(len(s['subsections']) for s in plan['sections'])} subsections | store {len(store)} collections")

In [ ]:
STYLE_GUIDE = (
 "STYLE: formal regulatory disclosure prose; concise; no marketing language; present figures with their "
 "units and reporting year; integrate IFRS S1 and S2 into one narrative where both apply (do not repeat)."
)  # <- inject your V9.7 style guide text here

MODE_INSTRUCTION = {
 "data_backed": ("Write the disclosure from the evidence. EVERY figure you state MUST be immediately followed "
                 "by its citation in square brackets, e.g. [E-BANK01-0037]. Never state a number that is not "
                 "in the evidence list. Prefer [primary] items for headline figures."),
 "absence":     ("The required data is unavailable. State the absence following the gap instruction verbatim. "
                 "Do NOT report missing values as zero and do NOT invent figures."),
 "narrative":   ("No quantitative data applies. Write a brief qualitative compliance statement addressing the "
                 "requirement(s). Use NO numbers and cite no evidence."),
}

def build_messages(block, section_title):
    reqs = "\n".join(
        f"- [{r['requirement_id']}] ({r['standard']} \u00b6{r['paragraph_id']}{r['clause_path'] or ''}): {r['requirement_text']}"
        for r in block["requirements"])
    def ev_val(e):
        v = e["value"]; return json.dumps(v, ensure_ascii=False) if isinstance(v, (dict, list)) else v
    evidence = "\n".join(
        f"- {e['citation_id']}: {ev_val(e)} {e['unit'] or ''} ({e['field']}, {e['reporting_year']})"
        f"{' [primary]' if e['is_primary'] else ''}"
        for e in block["evidence"]) or "(no evidence)"
    gaps = "\n".join(f"- {g['field']}: {g.get('instruction','')}" for g in block.get("data_gaps", []))
    system = ("You are an expert sustainability-reporting writer preparing a bank's IFRS S1 and IFRS S2 aligned "
              "climate-related disclosure. You write only what the requirements ask for and only what the evidence "
              "supports. You never invent figures. You cite every figure.")
    user = f"""SECTION: {section_title} > {block['subsection_title']}
CONCEPT: {block['concept']}    STANDARDS: {', '.join(block['standards_present'])}    MODE: {block['generation_mode']}

REQUIREMENTS (produce ONE aligned disclosure satisfying all of them; where IFRS S1 and IFRS S2 both appear, integrate into a single narrative citing both standards):
{reqs}

EVIDENCE (cite ONLY these citation_ids; each maps to a stored, audited value):
{evidence}
{('GAPS:' + chr(10) + gaps) if gaps else ''}
INSTRUCTION: {MODE_INSTRUCTION[block['generation_mode']]}
{STYLE_GUIDE}

Return ONLY a JSON object, no prose outside it:
{{"prose": "disclosure text with inline [E-...] citations after every figure",
  "citations_used": ["E-..."],
  "numeric_claims": [{{"text": "<the number and unit exactly as written in prose>", "citation_id": "E-..."}}],
  "requirements_addressed": ["<requirement_id>", "..."],
  "standards_covered": ["IFRS S1", "IFRS S2"]}}"""
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]

In [ ]:
def _to_number(s):
    m = re.search(r"-?\d[\d,]*\.?\d*", str(s))
    return float(m.group(0).replace(",", "")) if m else None

def gate(block, out):
    """Re-resolve every citation to the store; reject invented ids and mismatched numbers."""
    ev = {e["citation_id"]: e for e in block["evidence"]}
    fails = []
    for c in out.get("citations_used", []):
        if c not in ev:
            fails.append(f"cites unknown citation_id {c}")
    for nc in out.get("numeric_claims", []):
        cid = nc.get("citation_id")
        if cid not in ev:
            fails.append(f"numeric claim cites unknown id {cid}"); continue
        claimed, stored = _to_number(nc.get("text")), _to_number(ev[cid]["value"])
        if claimed is None or stored is None:
            continue
        tol = max(abs(stored) * 0.005, 0.01)           # 0.5% tolerance for rounding
        if abs(claimed - stored) > tol:
            fails.append(f"number '{nc.get('text')}' != evidence {ev[cid]['value']} for {cid}")
        # verify the store still backs the citation (pointer integrity at write time)
        if deref(ev[cid]["path"]) != ev[cid]["value"]:
            fails.append(f"evidence pointer drift for {cid}")
    if block["generation_mode"] == "narrative" and out.get("numeric_claims"):
        fails.append("narrative block must contain no numeric claims")
    if block["generation_mode"] == "data_backed" and block["evidence"] and not out.get("citations_used"):
        fails.append("data_backed block produced no citations")
    return (len(fails) == 0, fails)

In [ ]:
def generate_block(block, section_title, max_revisions=MAX_REVISIONS):
    messages = build_messages(block, section_title)
    last_out, last_fails = None, ["no output"]
    for attempt in range(max_revisions + 1):
        raw = azure_chat(messages)
        try:
            out = parse_contract(raw)
        except Exception as e:
            messages += [{"role": "assistant", "content": raw},
                         {"role": "user", "content": f"That was not valid JSON ({e}). Return ONLY the JSON object."}]
            last_fails = [f"invalid JSON: {e}"]; continue
        ok, fails = gate(block, out)
        if ok:
            return {"ok": True, "attempts": attempt, "output": out, "failures": []}
        last_out, last_fails = out, fails
        messages += [{"role": "assistant", "content": raw},
                     {"role": "user", "content": "Fix ALL of these and return corrected JSON only:\n- " + "\n- ".join(fails)}]
    return {"ok": False, "attempts": max_revisions, "output": last_out, "failures": last_fails}

In [ ]:
def run_writer(plan, blocks_by_id):
    generated, report_stats = {}, {"ok": 0, "failed": 0, "attempts": 0}
    for s in plan["sections"]:
        for ss in s["subsections"]:
            for bref in ss["blocks"]:
                block = blocks_by_id[bref["block_id"]]
                res = generate_block(block, s["section_title"])
                generated[bref["block_id"]] = res
                report_stats["attempts"] += res["attempts"]
                report_stats["ok" if res["ok"] else "failed"] += 1
                flag = "" if res["ok"] else f"  !! {res['failures']}"
                print(f"  {bref['block_id']:44s} attempts={res['attempts']} ok={res['ok']}{flag}")
    return generated, report_stats

def assemble(plan, generated):
    L = [f"# IFRS S1 & S2 Aligned Climate-Related Disclosure \u2014 {plan['bank_id']}", ""]
    for s in plan["sections"]:
        L += [f"## {s['order']}. {s['section_title']}", ""]
        for ss in s["subsections"]:
            L += [f"### {s['order']}.{ss['subsection_order']} {ss['subsection_title']}", ""]
            for bref in ss["blocks"]:
                res = generated.get(bref["block_id"])
                if res and res["output"]:
                    L += [res["output"]["prose"].strip(), ""]
    return "\n".join(L)

generated, stats = run_writer(plan, blocks_by_id)
print("\nblocks:", stats)
report_md = assemble(plan, generated)
(OUT / f"report_{BANK}.md").write_text(report_md, encoding="utf-8")
# audit sidecar: every citation used across the report, resolvable to a pointer
audit = [{"block_id": bid, "ok": r["ok"], "attempts": r["attempts"],
          "citations": (r["output"] or {}).get("citations_used", []), "failures": r["failures"]}
         for bid, r in generated.items()]
(OUT / f"report_{BANK}_audit.json").write_text(json.dumps(audit, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"wrote {OUT}/report_{BANK}.md  ({len(report_md)} chars) + audit sidecar")